In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from rockyclickup.wrapper import Session
from rockyclickup.models import Client

from rockyclickup.utils import response_to_dataframe

In [3]:

plan_types = [
    "fsa",
    "dca",
    "hra",
    "hsa",
    "pkg",
    "trn",
    "lsa",
    "ado",
    "edu",
]


In [4]:
clickup = Session()

In [5]:
client = Client(id="86877zwgd") # 86877zj7n


The following fields are not recorded in the rockyclickup database:
     FIELD ID                              |  FIELD NAME
     702d85f6-7155-447c-9019-206db17ab27c  |  Update Account Managers


In [6]:
client.client_contacts

['868bxrvg0', '868bxru9t', '868776z2n', '8687cehyr']

In [8]:

contact_ids = client.client_contacts

cu_contact_dictionaries = []
for cid in contact_ids:
    cu_res = clickup.get(endpoint=f"{clickup.base}/task/{cid}")

    cu_dict = cu_res.json()

    cu_contact_dictionaries.append(cu_dict)

cu_contact_df = response_to_dataframe(cu_contact_dictionaries)


In [ ]:
cu_contact_df[[
    'id',
    'custom_id',
    'custom_item_id',
    'name',
    'text_content',
    'description',
    'archived',
    'assignees',
    'tags',
    'parent',
    'top_level_parent',
    'priority',
    'due_date',
    'start_date',
    'points',
    'time_estimate',
    'time_spent',
    'dependencies',
    'linked_tasks',
    'locations',
    'team_id',
    'url',
    'permission_level',
    'attachments',
    'status.id',
    'status.status',
    'status.color',
    'status.orderindex',
    'status.type',
    'email_campaign_exclude',
    'phone_direct',
    'phone_direct_ext',
    'phone_mobile',
    'phone_fax',
    'name_first',
    'name_preferred',
    'name_last',
    'name_last_prior',
    'phone',
    'role',
    'owner',
    'agent_type',
    'client_contacts',
    'inv_contact',
    'fund_contact',
    'brokerage',
    'opportunity_broker',
    'email',
    'type',
    'role_tags',
    'title'
]]

In [10]:
cu_contact_df.columns.to_list()

['id',
 'custom_id',
 'custom_item_id',
 'name',
 'text_content',
 'description',
 'date_created',
 'date_updated',
 'date_closed',
 'date_done',
 'archived',
 'assignees',
 'group_assignees',
 'watchers',
 'checklists',
 'tags',
 'parent',
 'top_level_parent',
 'priority',
 'due_date',
 'start_date',
 'points',
 'time_estimate',
 'time_spent',
 'dependencies',
 'linked_tasks',
 'locations',
 'team_id',
 'url',
 'permission_level',
 'attachments',
 'status.id',
 'status.status',
 'status.color',
 'status.orderindex',
 'status.type',
 'creator.id',
 'creator.username',
 'creator.color',
 'creator.email',
 'creator.profilePicture',
 'list.id',
 'list.name',
 'list.access',
 'email_campaign_exclude',
 'phone_direct',
 'phone_direct_ext',
 'phone_mobile',
 'phone_fax',
 'name_first',
 'name_preferred',
 'name_last',
 'name_last_prior',
 'phone',
 'role',
 'owner',
 'social_linkedin',
 'social_facebook',
 'social_twitter',
 'address_mail',
 'agent_type',
 'client_contacts',
 'inv_contact',


In [ ]:
[f for f in dir(client) if "address" in f]

In [ ]:
[r for r in dir(client) if r.startswith("client_")]

In [ ]:
client.client_dca

In [ ]:

def _load_clickup_plans(client):
    possible_relation_fields = [
        f"client_{pt}" for pt in plan_types
    ]

    relation_fields = [
        rf for rf in dir(client) if rf in possible_relation_fields
    ]

    print(f"relation_fields:\n{relation_fields}")

    plan_ids = []
    for rf in relation_fields:
        print(rf)
        pids = getattr(client, rf)
        print(pids)
        plan_ids.extend(pids)

    plan_ids = list(set(plan_ids))

    print(f"plan_ids:\n{plan_ids}")

    cu_plan_responses = []
    for pid in plan_ids:
        cu_res = clickup.get(endpoint=f"{clickup.base}/task/{pid}")
        cu_plan_responses.append(cu_res)


    print(len(cu_plan_responses))

    return cu_plan_responses


In [ ]:
plans_responses = _load_clickup_plans(client)

In [ ]:
for plan in plans_responses:
    print(plan.json().get("id"))

In [ ]:
plan_dicts = [p.json() for p in plans_responses]
plan_df = response_to_dataframe(plan_dicts)

In [ ]:
plan_df

In [ ]:
sorted(plan_df.columns.to_list())

In [ ]:
necessary_columns = [
    "id",
    "name",
    "account_manager",
    "status.status",
    "elv_plan_code",
    "list.name",
    # "plan_year",
    "date_plan_start",
    "date_plan_end",
    "card",
    "annual_election_auto_adjust",
    "annual_election_max",
    "annual_election_min",
    "grace_period",
    "grace_termed_ee",
    "grace_termed_ee_days",
    "run_out",
    "run_out_termed_ee",
    "run_out_termed_ee_matches_plan",
    "elv_id",
]

narrow_df = plan_df[necessary_columns]

In [ ]:
plan_dicts = [row.to_dict() for _, row in narrow_df.iterrows()]

In [ ]:
plan_dicts